# Semana 01 · Ciclo de vida completo de un modelo con MLflow

En esta práctica llevarás un caso de clasificación desde los datos hasta una API local:

`datos → experimentos → comparación → gate → modelo ganador → Registry → API → observabilidad`

El objetivo no es conseguir el número más alto, sino construir una decisión **reproducible,
auditable y desplegable**. Usaremos Wine Quality como caso didáctico de clasificación
binaria: la etiqueta es 1 si `quality >= 6`; no debe interpretarse como garantía comercial.

**Entorno:** Databricks Free Edition, compute serverless y experimento asociado a este notebook.
No necesitas tarjeta de crédito, cluster propio, Docker, `uv`, `pyproject.toml` ni tokens.

## Mapa y criterios de terminación

Al acabar debes poder enseñar en MLflow:

1. un lote de al menos siete runs comparables, con datos, parámetros, métricas y artefactos;
2. una regla de selección escrita **antes** de mirar el test;
3. un único candidato evaluado sobre test y marcado como ganador;
4. versiones en Unity Catalog y un ciclo `Challenger → Champion → rollback`;
5. una API HTTP local que carga el `model.joblib` del run ganador y supera pruebas válidas e inválidas;
6. un run de despliegue con latencia, volumen de peticiones, errores y evidencia de la respuesta.

Tiempo recomendado: 45 min de construcción guiada + 120–180 min de práctica autónoma y debrief.

## 0. Dependencias

Ejecuta `%pip` al principio. Si Databricks reinicia Python, continúa desde la celda de imports.
Las dependencias se instalan en la sesión del notebook; no se crea ningún entorno o fichero de
proyecto en el repositorio.

In [ ]:
%pip install -q --upgrade "mlflow[databricks]==3.14.0" "scikit-learn==1.9.0" "pandas==2.2.3" "matplotlib==3.11.1" "joblib==1.5.3" "xgboost==3.0.5"

## Cómo trabajar con la versión sin resolver

Ejecuta las celdas en orden y completa cada `TODO`. Los bloques se detienen con
`NotImplementedError` para que un fallo no genere evidencia engañosa. Consulta la guía del
alumno antes de mirar la solución. Mantén el test cerrado hasta haber elegido el ganador.

## 1. Configuración e identidad del lote

In [ ]:
import json
import joblib
import re
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
import xgboost
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

STUDENT_ALIAS = "TODO-alias-anonimo"
BATCH_ID = uuid.uuid4().hex[:8]
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15
assert abs(TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE - 1.0) < 1e-9
MIN_VALIDATION_F1_MACRO = 0.70
QUALITY_THRESHOLD = 6
# Esta ruta es relativa a semana1/modules/01-mlflow-databricks-foundations/notebooks/.
DATASET_PATH = "../../../data/raw/WineQT.csv"

# TODO 1: valida el alias con una expresión regular y configura el tracking de Databricks.
raise NotImplementedError("TODO 1 · configuración")

## 2. Contrato y controles de datos

Permite nulos porque el pipeline los imputará, pero detente si faltan columnas, el dataset está
vacío, falta el target o `Id` se cuela como predictor. Construye un `quality_report`
serializable como JSON y confirma que el target contiene al menos dos clases.

In [ ]:
data = pd.read_csv(DATASET_PATH)
TARGET = "quality"
EXPECTED_FEATURES = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol",
]

# TODO 2: valida el contrato y crea X, y y quality_report.
# Hint: excluye `Id` y crea la etiqueta binaria 1 si `quality >= QUALITY_THRESHOLD`. Registra
# filas, duplicados, distribución, regla del umbral, nulos y checks ejecutados.
raise NotImplementedError("TODO 2 · calidad de datos")

## 3. Divide 70 % train, 15 % validación y 15 % test, siempre estratificado

In [ ]:
# TODO 3: crea X_train, X_valid, X_test, y_train, y_valid e y_test.
# Hint: aparta primero TEST_SIZE=0.15 con stratify=y. Sobre el 85 % restante, usa
# VALIDATION_SIZE / (TRAIN_SIZE + VALIDATION_SIZE) = 0.15 / 0.85 para obtener validación.
# Mantén RANDOM_STATE y stratify en las dos llamadas; no uses X_test ni y_test hasta la sección 9.
raise NotImplementedError("TODO 3 · split")

## 4. Pipeline y funciones de evaluación

In [ ]:
def build_pipeline(config: dict) -> Pipeline:
    # TODO 4a: usa un Pipeline con SimpleImputer(median) y el estimador indicado por
    # config['model_type']: ExtraTreesClassifier o XGBClassifier.
    # Hint: copia config antes de quitar model_type; las demás claves deben llegar al
    # estimador correspondiente. Conserva preprocessing y modelo en el mismo Pipeline
    # para que training y serving compartan transformaciones.
    raise NotImplementedError("TODO 4a · pipeline")


def evaluate_classifier(model: Pipeline, features: pd.DataFrame, labels: pd.Series) -> dict:
    # TODO 4b: calcula accuracy, precision_macro, recall_macro, f1_macro y latencia por fila.
    # Hint: usa average='macro' y zero_division=0; las dos clases son 0 y 1.
    raise NotImplementedError("TODO 4b · métricas")

## 5. Diseña al menos siete candidatos comparables

In [ ]:
# TODO 5: crea al menos siete configuraciones comparables. Incluye el candidato ExtraTrees
# que usa n_estimators=300, max_depth=None, min_samples_leaf=1, max_features=1.0,
# class_weight=None y random_state=RANDOM_STATE: es la configuración que supera el gate
# en la solución local. Añade también al menos un candidato con model_type='xgboost'.
# Hint XGBoost: usa objective='binary:logistic', eval_metric='logloss', tree_method='hist',
# learning_rate, max_depth, subsample, colsample_bytree, n_estimators y semillas explícitas.
# El primer candidato de la solución y de tools/local_wine_training.py usa 300 árboles,
# max_depth=None y min_samples_leaf=1; comprueba su F1 macro sólo sobre validación.
CANDIDATES = []
assert len(CANDIDATES) >= 7, "Necesitas al menos siete candidatos, incluyendo XGBoost."
assert any(candidate.get("model_type") == "xgboost" for candidate in CANDIDATES)

## 6. Tracking profundo: un run por candidato

Por cada candidato registra tags, parámetros, versiones, inputs de train/validación, tarjeta de
datos, calidad, riesgos, métricas, reporte por clase, matriz de confusión, modelo MLflow con
firma/input example, `deployment/model.joblib` y contrato de inferencia.

In [ ]:
dataset_card = {
    "source": "TODO",
    "purpose": "TODO",
    "features": EXPECTED_FEATURES,
    "target": TARGET,
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15, "random_state": RANDOM_STATE},
    "known_limitations": ["TODO", "TODO", "TODO"],
}
risk_register = {
    "version": "s01-lifecycle",
    "risks": [
        {"id": "R1", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R2", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R3", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R4", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
    ],
}
assert len(dataset_card["known_limitations"]) >= 3
assert len(risk_register["risks"]) >= 4

### 6a. Inputs de datos

Crea dos objetos `mlflow.data` con nombres distintos. Comprueba sus nombres antes de lanzar runs.

In [ ]:
# TODO 6a: crea train_dataset y validation_dataset con mlflow.data.from_pandas.
# Incluye el target alineado como `quality_high`, usa DATASET_PATH como source y conserva
# el mismo nombre de columnas en ambos inputs.
raise NotImplementedError("TODO 6a · datasets de MLflow")

### 6b. Artefactos del candidato

Implementa una función pequeña que registre firma/input example, `model_info.model_uri`, Joblib,
tamaño, matriz de confusión y contrato. Podrás probarla dentro de un único run antes del bucle.

In [ ]:
def log_candidate_artifacts(
    model: Pipeline, predictions: np.ndarray, candidate_index: int
) -> str:
    # TODO 6b: log_model(name='model'), tag candidate.model_uri y serializa con
    # joblib.dump en `model.joblib`. Regístralo con log_artifact en `deployment/`,
    # junto a joblib_size_bytes, matriz de confusión e inference_contract.json.
    # Hint: una Pipeline que contiene XGBClassifier se registra con la flavor sklearn
    # porque el artefacto incluye también el imputador. En MLflow 3.14, investiga
    # `skops_trusted_types` y permite sólo los tipos XGBoost que hayas revisado; no
    # desactives la protección para cargar tipos arbitrarios.
    # Devuelve model_uri y comprueba en MLflow que aparece el .joblib.
    raise NotImplementedError("TODO 6b · artefactos")

### 6c. Runs comparables

Ahora el bucle se ocupa sólo de identidad, parámetros, inputs, gobierno, entrenamiento, métricas
y de llamar al helper de artefactos. Conserva `experiment_id` y cada `run_id`.

In [ ]:
experiment_id = None
candidate_run_ids = []

# TODO 6c: implementa un start_run por configuración, registra tags/params/inputs/JSON,
# entrena, calcula validation.* para la etiqueta binaria, classification_report y llama
# log_candidate_artifacts. Hint: usa labels ordenadas para una matriz consistente y
# registra model_type y la versión de xgboost para poder explicar la comparación.
raise NotImplementedError("TODO 6c · runs de candidatos")
assert len(candidate_run_ids) == len(CANDIDATES)

## 7. Busca y compara exclusivamente los runs de tu alias y `BATCH_ID`

In [ ]:
# TODO 7: usa mlflow.search_runs con tres tags en filter_string y muestra parámetros,
# métricas de validación, latencia y run_id en un DataFrame `comparison`.
raise NotImplementedError("TODO 7 · comparación")

## 8. Aplica el gate sin mirar test

Filtra por F1 macro mínimo; ordena por F1 macro DESC, accuracy DESC y latencia ASC. Conserva
`BEST_RUN_ID`, `BEST_MODEL_URI` y un `selection_report` que diga explícitamente que test no
participó en la selección.

In [ ]:
# TODO 8: aplica el gate sobre validation.f1_macro >= MIN_VALIDATION_F1_MACRO.
# Hint: no abras test; si no hay elegibles, falla sin rebajar el umbral a posteriori.
raise NotImplementedError("TODO 8 · gate y ganador")

## 9. Carga el modelo desde MLflow, evalúa test una sola vez y completa el run ganador

In [ ]:
# TODO 9: load_model(BEST_MODEL_URI), calcula test.*, reabre BEST_RUN_ID,
# registra el input de testing, métricas, tags de selección y selection/decision.json.
raise NotImplementedError("TODO 9 · evaluación final")

## 10. Registra el modelo ganador en Unity Catalog

Usa el catálogo y esquema activos, un nombre de tres niveles y tu alias saneado. Añade
descripción y tags. Registra referencia+ganador, asigna `Challenger`, haz smoke test,
promociona, comprueba rollback y deja el ganador como `Champion`. Si recibes `PERMISSION_DENIED`, consulta la
guía: no pegues credenciales ni cambies a recursos de pago.

In [ ]:
# TODO 10: set_registry_uri('databricks-uc'), register_model, descripciones,
# tags, Challenger, promoción, rollback, re-promoción y variable champion.
raise NotImplementedError("TODO 10 · Model Registry")

## 11. Carga `models:/<nombre>@Champion` y realiza un smoke test con tres filas

In [ ]:
# TODO 11: prueba que el consumidor puede cargar por alias y predecir tres filas.
raise NotImplementedError("TODO 11 · smoke test del Registry")

## 12. API local con el artefacto Joblib ganador

Resuelve el run de `Champion`, descarga `deployment/model.joblib` y levanta un
`ThreadingHTTPServer` en `127.0.0.1` con puerto automático. Implementa `GET /health` y
`POST /predict`; valida `instances`, columnas ausentes y columnas extra.

In [ ]:
# TODO 12a: resuelve champion.run_id, descarga runs:/.../deployment/model.joblib,
# cárgalo con joblib.load y comprueba que predice una fila con EXPECTED_FEATURES.
raise NotImplementedError("TODO 12a · cargar artefacto Joblib")

### 12b. Handler y contrato

Define `PredictionHandler` con `_send_json`, `GET /health` y `POST /predict`. Valida objeto raíz,
lista no vacía de objetos, columnas exactas y errores 400. Instrumenta contadores y latencias.

In [ ]:
API_STATS = {"requests_total": 0, "requests_success": 0, "requests_rejected": 0, "requests_error": 0, "latencies_ms": []}
API_STATS_LOCK = threading.Lock()

# TODO 12b: define PredictionHandler y actualiza API_STATS en cada respuesta.
raise NotImplementedError("TODO 12b · handler")

### 12c. Arranque y limpieza idempotente

Arranca en `127.0.0.1` y puerto 0. Define `stop_api()` antes de probar para que un `finally` pueda
cerrar servidor, socket e hilo. No uses `0.0.0.0`.

In [ ]:
# TODO 12c: crea api_server, API_PORT, api_thread, API_STOPPED y stop_api().
raise NotImplementedError("TODO 12c · ciclo de servidor")

## 13. Pruebas de contrato y de error

In [ ]:
# TODO 13: crea call_json con try/finally para apagar siempre la API y verifica:
# GET /health -> 200; POST válido -> 200; sólo alcohol -> 400; JSON no-object -> 400.
raise NotImplementedError("TODO 13 · pruebas de API")

## 14. Registra un run separado para la evidencia del despliegue

In [ ]:
# TODO 14: registra versión/alias/run origen, requests totales, éxitos, rechazadas,
# latencias y deployment/evidence.json. Conserva DEPLOYMENT_RUN_ID.
# No registres el payload de entrada: la evidencia debe contener sólo agregados.
raise NotImplementedError("TODO 14 · observabilidad de deployment")

## 15. Apaga la API y entrega el resumen del ciclo

In [ ]:
# TODO 15: shutdown, server_close, join y assert de que el hilo terminó.
# Imprime experiment_id, batch, ganador, modelo/versión/alias y deployment run.
raise NotImplementedError("TODO 15 · cierre")